**Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,confusion_matrix,
                             classification_report
)

**Load Dataset**

In [ ]:
nltk.download("stopwords")

**Load Dataset**

In [ ]:
data = pd.read_csv("fake_and_real_news.csv")

,Text,label
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake
1,U.S. conservative leader optimistic of common ...,Real
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real
3,Court Forces Ohio To Allow Millions Of Illega...,Fake
4,Democrats say Trump agrees to work on immigrat...,Real


**Data Exploration**

In [ ]:
data.head()

In [ ]:
data.shape()

In [ ]:
data.columns()

In [ ]:
data.info()

In [ ]:
data["label"].value_counts()

In [ ]:
print(data["Text"][0])

**Text preprocessing**

In [ ]:
data["Text"] = data["Text"].str.lower()

In [ ]:
def remove_punctuation(text):
    for char in string.punctuation:
        text = text.replace(char, "")
    return text

In [ ]:
data["Text"] = data["Text"].apply(remove_punctuation)

In [ ]:
stop_words = set(stopwords.words("english"))

In [ ]:
def tokenize(text):
    return text.split()

In [ ]:
def remove_stopwords(words):
    filtered_words = []

    for word in words:
        if word not in stop_words:
            filtered_words.append(word)

    return filtered_words

In [ ]:
stemmer = PorterStemmer()

In [ ]:
def stem_words(words):
    stemmed_words = []

    for word in words:
        stemmed_words.append(stemmer.stem(word))

    return stemmed_words

In [ ]:
def preprocess(text):

    text = text.lower()

    text = remove_punctuation(text)

    words = tokenize(text)

    words = remove_stopwords(words)

    words = stem_words(words)

    return " ".join(words)

In [ ]:
data["Text"] = data["Text"].apply(preprocess)

**TF-IDF Vectorization**

In [ ]:
X = data["Text"]
y = data["label"]

In [ ]:
vectorizer = TfidfVectorizer()

**Train-Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

**Train Logistic Regression**

In [ ]:
model = LogisticRegression()

In [ ]:
model.fit(X_train, y_train)

**Evaluate Model**

In [ ]:
prediction = model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, prediction)

print("Accuracy:", accuracy)

In [ ]:
cm = confusion_matrix(y_test, prediction)

print(cm)

In [ ]:
print(classification_report(y_test, prediction))

In [ ]:
import pickle

In [ ]:
with open("../models/model.pkl", "wb") as file:
    pickle.dump(model, file)

In [ ]:
with open("../models/vectorizer.pkl", "wb") as file:
    pickle.dump(vectorizer, file)

**Predict on Custom News**

In [ ]:
news = input("Enter News: ")

In [ ]:
clean_news = preprocess(news)

In [ ]:
vector = vectorizer.transform([clean_news])

In [ ]:
prediction = model.predict(vector)

In [ ]:
if prediction[0] == "Real":
    print("Real News")
else:
    print("Fake News")